In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import random

# Upload the file
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

Saving placement.csv to placement (3).csv
Uploaded file: placement (3).csv


## Step 2: Load and Preprocess Data


In [ ]:
# Determine file type and load data
if filename.endswith('.csv'):
    df = pd.read_csv(filename)
elif filename.endswith(('.xls', '.xlsx')):
    df = pd.read_excel(filename)
else:
    raise ValueError("Unsupported file format. Please upload a .csv or .xlsx file.")

print("Original DataFrame head:")
display(df.head())
print("\nOriginal DataFrame info:")
df.info()

# Handle missing values by filling with the mean (for numerical columns)
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].mean())

# Separate features (X) and target (y)
X = df.iloc[:, :-1] # All columns except the last one
y = df.iloc[:, -1]  # The last column

# Convert categorical features to numerical using one-hot encoding if any
X = pd.get_dummies(X, drop_first=True)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add intercept term to X_train_scaled and X_test_scaled
X_train_scaled = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_test_scaled = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

print("\nData preprocessing complete. Shapes:")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test_scaled: {X_test_scaled.shape}")
print(f"y_test: {y_test.shape}")

Original DataFrame head:


,cgpa,package
0,6.89,3.26
1,5.12,1.98
2,7.82,3.25
3,7.42,3.67
4,6.94,3.57



Original DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cgpa     200 non-null    float64
 1   package  200 non-null    float64
dtypes: float64(2)
memory usage: 3.3 KB

Data preprocessing complete. Shapes:
X_train_scaled: (160, 2)
y_train: (160,)
X_test_scaled: (40, 2)
y_test: (40,)


## Step 3: Implement Gradient Descent Algorithms

Below are the implementations for Batch, Stochastic, and Mini-Batch Gradient Descent. Each function will train a linear regression model and return the learned coefficients (including intercept) and a history of the cost function during training.

In [ ]:
# 1. Batch Gradient Descent (BGD)
class BGD:

  def __init__(self, lr = 0.01, epochs = 100):
    self.lr = lr
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self, X, y):
    self.intercept_ = 0
    self.coef_ = np.ones(X.shape[1])

    for i in range(self.epochs):
      # intercept
      #Vectorization - Makinng changes wrt to *Error*
      y_hat = np.dot(X, self.coef_) + self.intercept_
      intercept_der = -2 * np.mean(y-y_hat)
      self.intercept_ = self.intercept_ - (self.lr * intercept_der)

      # coeff
      coef_der = -2 * np.dot((y-y_hat), X)
      self.coef_ = self.coef_ - (self.lr * coef_der)

    print(self.intercept_)
    print(self.coef_)

  def predict(self, X):
    return np.dot(X, self.coef_) + self.intercept_   # y_pred = mx + b

# 2. Stochastic Gradient Descent (SGD)
class SGD:

  def __init__(self, lr = 0.01, epochs = 100):
    self.lr = lr
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None

  def fit(self, X, y):
    self.intercept_ = 0
    self.coef_ = np.ones(X.shape[1])

    for i in range(self.epochs):
      idx = np.random.randint(0, X.shape[0]) # Moved inside the loop
      # intercept
      #Vectorization - Makinng changes wrt to *Error*
      y_hat = np.dot(X[idx], self.coef_) + self.intercept_
      intercept_der = -2 *(y[idx]-y_hat)
      self.intercept_ = self.intercept_ - (self.lr * intercept_der)

      # coeff
      coef_der = -2 * (y[idx]-y_hat) * X[idx] # Corrected to use X[idx]
      self.coef_ = self.coef_ - (self.lr * coef_der)

    print(self.intercept_)
    print(self.coef_)

  def predict(self, X):
    return np.dot(X, self.coef_) + self.intercept_   # y_pred = mx + b

# 3. Mini-Batch Gradient Descent (MBGD)
class MBGD:

  def __init__(self,batch_size = 10, lr = 0.01, epochs = 100):
    self.lr = lr
    self.epochs = epochs
    self.coef_ = None
    self.intercept_ = None
    self.batch_size = batch_size

  def fit(self, X, y):
    self.intercept_ = 0.0 # Initialize as float
    self.coef_ = np.ones(X.shape[1])

    for i in range(self.epochs):
      for j in range(int(X.shape[0]/self.batch_size)):
        # Generate batch indices
        idx = random.sample(range(X.shape[0]), self.batch_size)

        X_batch = X[idx]
        y_batch = y[idx]

        # intercept
        y_hat = np.dot(X_batch, self.coef_) + self.intercept_
        intercept_der = -2 * np.mean(y_batch-y_hat) # Average over batch
        self.intercept_ = self.intercept_ - (self.lr * intercept_der)

        # coeff
        coef_der = -2 * np.dot((y_batch-y_hat), X_batch) # Use dot product for coefficients
        self.coef_ = self.coef_ - (self.lr * coef_der)

    print(self.intercept_)
    print(self.coef_)

  def predict(self, X):
    return np.dot(X, self.coef_) + self.intercept_   # y_pred = mx + b

## Step 4: Run and Display Results for Each Algorithm


In [ ]:
# Set hyperparameters
learning_rate = 0.001 # Reduced learning rate significantly
n_iterations = 20000 # Increased iterations for better convergence, especially with smaller learning rate
batch_size = 32 # For Mini-Batch GD

# Prepare data for custom GD classes (remove intercept column if the class handles it separately)
# X_train_scaled and X_test_scaled from cell 51cbfab9 already include the intercept at index 0.
# The custom GD classes also have a separate `intercept_` attribute.
# To avoid double-counting the intercept or misinterpreting the 'X' argument,
# we pass only the features (columns from index 1 onwards) and ensure y is a numpy array.
X_train_features = X_train_scaled[:, 1:]
X_test_features = X_test_scaled[:, 1:]
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()


# --- Batch Gradient Descent ---
print("### Batch Gradient Descent (BGD) ###")
# Instantiate with specified learning_rate and epochs (n_iterations)
bgd = BGD(lr=learning_rate, epochs=n_iterations)
bgd.fit(X_train_features, y_train_np)
y_pred_bgd = bgd.predict(X_test_features)
r2_bgd = r2_score(y_test_np, y_pred_bgd)
bgd_coef = bgd.coef_
bgd_intercept = bgd.intercept_

print(f"Coefficients: {bgd_coef}")
print(f"Intercept: {bgd_intercept}")
print(f"R2 Score (Test Set): {r2_bgd:.4f}")


# --- Stochastic Gradient Descent ---
print("\n### Stochastic Gradient Descent (SGD) ###")
# Instantiate with specified learning_rate and epochs (n_iterations)
sgd = SGD(lr=learning_rate, epochs=n_iterations)
sgd.fit(X_train_features, y_train_np)
y_pred_sgd = sgd.predict(X_test_features)
r2_sgd = r2_score(y_test_np, y_pred_sgd)
sgd_coef = sgd.coef_
sgd_intercept = sgd.intercept_

print(f"Coefficients : {sgd_coef}")
print(f"Intercept: {sgd_intercept}")
print(f"R2 Score (Test Set): {r2_sgd:.4f}")

# --- Mini-Batch Gradient Descent ---
print("\n### Mini-Batch Gradient Descent (MBGD) ###")
# Instantiate with specified learning_rate, epochs (n_iterations), and batch_size
mbgd = MBGD(batch_size=batch_size, lr=learning_rate, epochs=n_iterations)
mbgd.fit(X_train_features, y_train_np)
y_pred_mbgd = mbgd.predict(X_test_features)
r2_mbgd = r2_score(y_test_np, y_pred_mbgd)
mbgd_coef = mbgd.coef_
mbgd_intercept = mbgd.intercept_

print(f"Coefficients: {mbgd_coef}")
print(f"Intercept: {mbgd_intercept}")
print(f"R2 Score (Test Set): {r2_mbgd:.4f}") # Fixed typo here


print("\n--- Summary of R2 Scores ---")
print(f"Batch Gradient Descent R2: {r2_bgd:.4f}")
print(f"Stochastic Gradient Descent R2: {r2_sgd:.4f}")
print(f"Mini-Batch Gradient Descent R2: {r2_mbgd:.4f}")

### Batch Gradient Descent (BGD) ###
2.9958749999998897
[0.62421668]
Coefficients: [0.62421668]
Intercept: 2.9958749999998897
R2 Score (Test Set): 0.7731

### Stochastic Gradient Descent (SGD) ###
2.982373031553067
[0.61830471]
Coefficients : [0.61830471]
Intercept: 2.982373031553067
R2 Score (Test Set): 0.7702

### Mini-Batch Gradient Descent (MBGD) ###
2.995403779789022
[0.62169398]
Coefficients: [0.62169398]
Intercept: 2.995403779789022
R2 Score (Test Set): 0.7733

--- Summary of R2 Scores ---
Batch Gradient Descent R2: 0.7731
Stochastic Gradient Descent R2: 0.7702
Mini-Batch Gradient Descent R2: 0.7733
